In [8]:
from src.mnist_model import MNIST_CNN
from src.training import get_accuracy
from src.utils import device
from settings import settings
from torch.utils.data import DataLoader
import torch

model = MNIST_CNN()
model.to(device)
model.load_state_dict(torch.load(settings.models_path / "mnist_model.pth"))
from mnist_training import test_data, test_labels, train_data
model.binary_mode()
print(get_accuracy(model, test_data, test_labels))
from src.improved_model import binary_sign

model.binary_mode()
layer1 = model.flatten(model.layer1.activation(model.layer1(train_data))).detach()
layer2 = model.layer3.activation(model.layer3(layer1)).detach()
layer3 = model.layer4.activation(model.layer4(layer2)).detach()


layer1 = layer1.cpu().numpy() == 1
layer2 = layer2.cpu().numpy() == 1
layer3 = layer3.cpu().numpy() == 1
weight = model.layer3.weight.detach().cpu().numpy() == 1


C:\Users\frrit\AppData\Local\Temp\ipykernel_27592\2400393949.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(settings.models_path / "mn

0.9057


In [9]:
row = 1

In [15]:
import cupy as cp  # Import CuPy for GPU acceleration

def best_threshold_vectorized_gpu(y, z):
    thresholds = cp.unique(z)  # Get unique threshold values

    preds = z[:, None] >= thresholds[None, :]
    errors = cp.sum(preds != y[:, None], axis=0)
    best_idx = cp.argmin(errors)
    best_t = thresholds[best_idx]
    accuracy = 1 - errors[best_idx] / len(y)

    return best_t.get(), accuracy.get()  # Move results back to CPU

# Move data to GPU
x = cp.asarray(layer1)
y = cp.asarray(layer2)[:, row]
weight = cp.asarray(weight)

n_features = x.shape[1]
A = weight[row, :]
z = cp.sum(~(cp.bitwise_xor(A, x)), axis=1)
best_threshold, best_accuracy = best_threshold_vectorized_gpu(y, z)

epochs = 5
for epoch in range(epochs):
    improved = False
    for index in cp.random.permutation(n_features):  # Use CuPy for randomness

        delta = (x[:, index] == ~A[index]).astype(cp.int8) - (x[:, index] == A[index]).astype(cp.int8)
        z += delta

        # Compute accuracy and threshold
        threshold, accuracy = best_threshold_vectorized_gpu(y, z)

        if accuracy > best_accuracy:
            A[index] = ~A[index]
            best_accuracy = accuracy
            best_threshold = threshold
            improved = True
            print(best_accuracy)
        else:
            z -= delta  # Undo if no improvement

    if not improved:
        break  # Stop early if no improvement


0.8694833333333334
0.8694999999999999
0.8695166666666667
0.86955
0.8695666666666666
0.8695833333333334
0.8696
0.8696166666666667
0.8696833333333334


In [10]:
np.bincount(y)

Computing row 1
